# Resultados del pronóstico municipal 6→6, clúster 2

[← Metodología común](metodologia-pronostico-6x6.ipynb)

Este notebook presenta exclusivamente la configuración y los resultados de
`cluster2`. El entrenamiento comienza en `2014-01-01` y el detalle inicial se
centra en **Santiago**.


In [9]:
# Configuración editable del notebook
from utils.forecasting_cache import ForecastCacheConfig

CLUSTER_COLUMN = "cluster2"
TRAINING_START = "2014-01-01"
PREFERRED_MUNICIPALITY = "Santiago"

MODEL_CACHE = ForecastCacheConfig(
    directory="model_cache",
    enabled=True,
    refresh=False,
)


## Composición del clúster 2

La celda resuelve las comunas marcadas en `cluster2` y deja visible la configuración
efectiva antes de cargar los datos.


In [10]:
import pandas as pd
from IPython.display import HTML, display

from utils.data import data_loader
from utils.forecasting_cache import summarize_cache_report
from utils.forecasting_workflow import (
    build_municipality_forecast_view,
    macro_micro_gap,
    resolve_cluster_municipalities,
    run_cluster_forecasting_workflow,
)
from utils.municipal_income import build_municipalities_income_history
from utils.notebook_display import collapsible_stdout, display_collapsible
from utils.plots import (
    plot_annual_income_share_lines,
    plot_municipality_forecast_evaluation,
)

clusters = pd.read_csv("clusters.csv")
municipalities = resolve_cluster_municipalities(clusters, CLUSTER_COLUMN)
display_collapsible(
    "Ver configuración efectiva",
    pd.DataFrame(
        {
            "cluster": [CLUSTER_COLUMN],
            "comunas": [len(municipalities)],
            "inicio de entrenamiento": [pd.Timestamp(TRAINING_START)],
            "comuna para detalle": [PREFERRED_MUNICIPALITY],
            "caché habilitado": [MODEL_CACHE.enabled],
            "refrescar caché": [MODEL_CACHE.refresh],
            "directorio caché": [str(MODEL_CACHE.directory)],
        }
    )
)
municipality_label = "comuna" if len(municipalities) == 1 else "comunas"
municipality_items = "".join(
    f"<li>{municipality}</li>" for municipality in municipalities
)
display_collapsible(
    f"Ver {len(municipalities)} {municipality_label} del clúster",
    HTML(
        f'<ul class="cluster-members__list">{municipality_items}</ul>'
    )
)


,cluster,comunas,inicio de entrenamiento,comuna para detalle,caché habilitado,refrescar caché,directorio caché
0,cluster2,25,2014-01-01,Santiago,True,False,model_cache


## Historia descriptiva del clúster 2

Se muestran trayectorias independientes para las comunas de `cluster2` antes de
presentar la evaluación predictiva.


In [11]:
with collapsible_stdout("Ver registro de carga de datos"):
    presupuesto_cluster = data_loader(municipalities=municipalities)
historial_cluster = build_municipalities_income_history(
    presupuesto_cluster,
    municipalities=municipalities,
)

for municipality in municipalities:
    municipality_history = historial_cluster.loc[
        historial_cluster["Nombre Municipio"].eq(municipality)
    ]
    figure = plot_annual_income_share_lines(municipality_history)
    year_count = int(municipality_history["Ejercicio"].nunique())
    figure.update_layout(width=max(900, year_count * 90 + 240), autosize=False)
    figure_html = figure.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        config={"responsive": False, "displaylogo": False},
    )
    display(
        HTML(
            '<div style="max-width:100%; overflow-x:auto; padding-bottom:1rem;">'
            f"{figure_html}</div>"
        )
    )


## Ejecución y trazabilidad del clúster 2

El bloque ejecuta el workflow conjunto para `cluster2`. El contrato temporal, las
métricas y el comportamiento del caché están documentados en la
[metodología común](metodologia-pronostico-6x6.ipynb).


In [12]:
with collapsible_stdout("Ver registro de entrenamiento conjunto"):
    workflow = run_cluster_forecasting_workflow(
        presupuesto_cluster,
        municipalities,
        cluster_label=CLUSTER_COLUMN,
        training_start=TRAINING_START,
        progress=True,
        cache_config=MODEL_CACHE,
    )

display_collapsible("Ver configuración del workflow", workflow.configuration)
display_collapsible("Ver ventanas de entrenamiento", workflow.training_windows)
display_collapsible("Ver progresión de modelos", workflow.model_progression)
display_collapsible(
    "Ver resumen del caché",
    summarize_cache_report(workflow.cache_report),
)

cache_by_stage = (
    workflow.cache_report.groupby(
        ["stage", "artifact_type", "status"], sort=False, dropna=False
    )
    .agg(artefactos=("key", "size"), gb=("size_bytes", lambda x: x.sum() / 1024**3))
    .reset_index()
)
display_collapsible("Ver artefactos de caché por etapa", cache_by_stage)


,cluster,comunas,inicio_entrenamiento_solicitado,primer_corte_tuning,ultimo_corte_tuning,primer_corte_validacion,ultimo_corte_validacion,fin_entrenamiento_final,entrada_final,test_congelado,ventanas_entrenamiento_final,variables_mensuales,objetivos_directos
0,cluster2,25,2014-01-01,2018-06-01,2022-12-01,2023-06-01,2024-06-01,2025-06-01,julio-diciembre 2025,enero-junio 2026,3175,30,30


,Nombre Municipio,primera_entrada,ultimo_objetivo,ventanas
0,Cerrillos,2014-01-01,2025-06-01,127
1,Cerro Navia,2014-01-01,2025-06-01,127
2,Curacaví,2014-01-01,2025-06-01,127
3,El Bosque,2014-01-01,2025-06-01,127
4,El Monte,2014-01-01,2025-06-01,127
5,Estación Central,2014-01-01,2025-06-01,127
6,Independencia,2014-01-01,2025-06-01,127
7,Isla de Maipo,2014-01-01,2025-06-01,127
8,La Cisterna,2014-01-01,2025-06-01,127
9,La Granja,2014-01-01,2025-06-01,127


,nivel,modelo,complejidad
0,1,Persistencia,Repite el último vector mensual; no se entrena.
1,2,Ridge global,Relación lineal regularizada; comuna y mes one...
2,3,ExtraTrees global,Ensamble no lineal; comuna y mes one-hot.
3,4,CatBoost global,Boosting no lineal; comuna y mes categóricos n...


,hits,misses,modelos_cargados,evaluaciones_cargadas,reutilizaciones_memoria,artefactos_escritos,artefactos_corruptos_aislados,gb_cargados,gb_escritos,gb_artefactos_utilizados
0,117,0,3,114,0,0,0,0.486808,0.0,0.486808


,stage,artifact_type,status,artefactos,gb
0,joint_tuning,evaluation,hit,110,0.001555
1,joint_validation,evaluation,hit,3,0.000088
2,joint_final_model,model,hit,3,0.485136
3,joint_test,evaluation,hit,1,0.000029


## Hiperparámetros elegidos para el clúster 2

La tabla identifica las configuraciones retenidas para `cluster2` antes de comparar
familias en validación.


In [13]:
tuning_table = workflow.tuning_results.copy()
tuning_table["WAPE macro (%)"] = (100 * tuning_table["wape_macro"]).round(2)
tuning_table["WAPE micro (%)"] = (100 * tuning_table["wape_micro"]).round(2)
tuning_table["MAE (MM CLP)"] = tuning_table["mae_mm_clp"].round(2)
selected_parameters = pd.DataFrame(
    [
        {
            "modelo": spec.name,
            "candidate_id": spec.candidate_id,
            "parametros": dict(spec.params),
        }
        for spec in workflow.selected_specs
    ]
)
display_collapsible(
    "Ver resultados de tuning",
    tuning_table[
        [
            "familia",
            "modelo_candidato",
            "parametros",
            "seleccionado",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
        ]
    ]
)
display_collapsible("Ver hiperparámetros seleccionados", selected_parameters)


,familia,modelo_candidato,parametros,seleccionado,WAPE macro (%),WAPE micro (%),MAE (MM CLP)
0,ridge,Ridge global [alpha=10],{'alpha': 10.0},True,21.65,20.28,530.52
1,ridge,Ridge global [alpha=1],{'alpha': 1.0},False,21.97,20.48,535.63
2,ridge,Ridge global [alpha=0.1],{'alpha': 0.1},False,22.06,20.54,537.20
3,extra_trees,"ExtraTrees global [depth=none, leaf=1]","{'n_estimators': 400, 'max_depth': None, 'min_...",True,21.00,19.95,521.68
4,extra_trees,"ExtraTrees global [depth=none, leaf=3]","{'n_estimators': 400, 'max_depth': None, 'min_...",False,21.48,20.34,532.09
5,extra_trees,"ExtraTrees global [depth=12, leaf=3]","{'n_estimators': 400, 'max_depth': 12, 'min_sa...",False,23.93,22.50,588.57
6,extra_trees,"ExtraTrees global [depth=12, leaf=1]","{'n_estimators': 400, 'max_depth': 12, 'min_sa...",False,24.28,22.76,595.23
7,catboost,"CatBoost global [depth=6, iterations=500]","{'iterations': 500, 'depth': 6, 'learning_rate...",True,22.91,21.19,554.32
8,catboost,"CatBoost global [depth=4, iterations=500]","{'iterations': 500, 'depth': 4, 'learning_rate...",False,23.90,22.13,578.76
9,catboost,"CatBoost global [depth=6, iterations=300]","{'iterations': 300, 'depth': 6, 'learning_rate...",False,24.17,22.38,585.24


,modelo,candidate_id,parametros
0,Ridge global,ridge_alpha_10,{'alpha': 10.0}
1,ExtraTrees global,extra_trees_depth_none_leaf_1,"{'n_estimators': 400, 'max_depth': None, 'min_..."
2,CatBoost global,catboost_depth_6_iterations_500,"{'iterations': 500, 'depth': 6, 'learning_rate..."


## Selección histórica del clúster 2

Este bloque muestra el ganador congelado de `cluster2` y sus métricas por grupo de
ingreso antes de abrir el test de 2026.


In [14]:
validation_ranking = workflow.validation_ranking.copy()
validation_ranking["WAPE macro (%)"] = (
    100 * validation_ranking["wape_macro"]
).round(2)
validation_ranking["WAPE micro (%)"] = (
    100 * validation_ranking["wape_micro"]
).round(2)

validation_metrics = workflow.validation_summary.copy()
validation_metrics["WAPE macro (%)"] = (
    100 * validation_metrics["wape_macro"]
).round(2)
validation_metrics["WAPE micro (%)"] = (
    100 * validation_metrics["wape_micro"]
).round(2)
validation_metrics["MAE (MM CLP)"] = validation_metrics["mae_mm_clp"].round(2)
validation_metrics["Sesgo (MM CLP)"] = (
    validation_metrics["sesgo_micro_mm_clp"].round(2)
)

display_collapsible(
    "Ver ranking de validación",
    validation_ranking[
        [
            "ranking",
            "seleccion_validacion",
            "modelo",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "mae_mm_clp",
            "sesgo_micro_mm_clp",
        ]
    ]
)
display_collapsible(
    "Ver métricas de validación por ingreso",
    validation_metrics[
        [
            "modelo",
            "grupo_ingreso",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
            "Sesgo (MM CLP)",
        ]
    ]
)
display_collapsible(
    "Ver modelo seleccionado por validación",
    workflow.selected_model_name,
)


,ranking,seleccion_validacion,modelo,WAPE macro (%),WAPE micro (%),mae_mm_clp,sesgo_micro_mm_clp
0,1,True,ExtraTrees global,19.45,18.91,647.506706,-362.274264
1,2,False,CatBoost global,21.56,20.37,697.692237,-293.584241
2,3,False,Ridge global,22.45,20.64,706.927985,-149.227974
3,4,False,Persistencia,44.05,43.36,1484.804413,-72.060067


,modelo,grupo_ingreso,WAPE macro (%),WAPE micro (%),MAE (MM CLP),Sesgo (MM CLP)
0,CatBoost global,FCM,24.16,23.16,332.53,-102.36
1,CatBoost global,IPP,29.08,24.80,365.87,-109.44
2,CatBoost global,Otros ingresos,66.26,43.74,125.54,-54.75
3,CatBoost global,Total disponible,21.56,20.37,697.69,-293.58
4,CatBoost global,Transferencias corrientes,105.63,96.24,150.60,-22.98
5,CatBoost global,Transferencias de capital,161.81,131.54,100.32,-4.05
6,ExtraTrees global,FCM,21.45,22.21,318.81,-156.36
7,ExtraTrees global,IPP,20.26,19.50,287.66,-181.71
8,ExtraTrees global,Otros ingresos,70.66,35.84,102.88,-36.27
9,ExtraTrees global,Total disponible,19.45,18.91,647.51,-362.27


## Test congelado de 2026 para el clúster 2

La comparación fuera de muestra de `cluster2` aplica la cobertura observada definida
en la [metodología común](metodologia-pronostico-6x6.ipynb).


In [15]:
test_ranking = workflow.test_ranking.copy()
test_ranking["WAPE macro (%)"] = (100 * test_ranking["wape_macro"]).round(2)
test_ranking["WAPE micro (%)"] = (100 * test_ranking["wape_micro"]).round(2)

test_metrics = workflow.test_summary.copy()
test_metrics["WAPE macro (%)"] = (100 * test_metrics["wape_macro"]).round(2)
test_metrics["WAPE micro (%)"] = (100 * test_metrics["wape_micro"]).round(2)
test_metrics["MAE (MM CLP)"] = test_metrics["mae_mm_clp"].round(2)
test_metrics["Sesgo (MM CLP)"] = test_metrics["sesgo_micro_mm_clp"].round(2)

display_collapsible(
    "Ver ranking del test 2026",
    test_ranking[
        [
            "ranking",
            "mejor_test",
            "modelo",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "mae_mm_clp",
            "sesgo_micro_mm_clp",
        ]
    ]
)
display_collapsible(
    "Ver métricas del test 2026 por ingreso",
    test_metrics[
        [
            "modelo",
            "grupo_ingreso",
            "WAPE macro (%)",
            "WAPE micro (%)",
            "MAE (MM CLP)",
            "Sesgo (MM CLP)",
            "meses_evaluados",
        ]
    ]
)

if len(workflow.municipalities) == 1:
    assert macro_micro_gap(workflow.test_summary).fillna(0).le(1e-12).all()


,ranking,mejor_test,modelo,WAPE macro (%),WAPE micro (%),mae_mm_clp,sesgo_micro_mm_clp
0,1,True,ExtraTrees global,16.35,15.83,542.812925,-124.902308
1,2,False,CatBoost global,19.95,18.88,647.348500,22.970032
2,3,False,Ridge global,23.46,21.90,750.779365,146.060298
3,4,False,Persistencia,38.95,43.06,1476.585472,-46.068208


,modelo,grupo_ingreso,WAPE macro (%),WAPE micro (%),MAE (MM CLP),Sesgo (MM CLP),meses_evaluados
0,CatBoost global,FCM,25.93,26.78,340.36,102.30,140.0
1,CatBoost global,IPP,23.63,21.82,342.59,-20.02,140.0
2,CatBoost global,Otros ingresos,90.19,39.63,137.87,-59.35,140.0
3,CatBoost global,Total disponible,19.95,18.88,647.35,22.97,140.0
4,CatBoost global,Transferencias corrientes,119.04,102.46,146.75,18.41,140.0
5,CatBoost global,Transferencias de capital,305.35,127.39,123.68,-18.37,140.0
6,ExtraTrees global,FCM,19.61,20.19,256.62,18.67,140.0
7,ExtraTrees global,IPP,17.40,15.77,247.56,-107.06,140.0
8,ExtraTrees global,Otros ingresos,113.66,36.58,127.26,-30.25,140.0
9,ExtraTrees global,Total disponible,16.35,15.83,542.81,-124.90,140.0


## Detalle inicial de Santiago

`PREFERRED_MUNICIPALITY` permite cambiar la comuna inspeccionada dentro de `cluster2`
sin alterar la selección ni el ranking ya calculados.


In [16]:
municipality_view = build_municipality_forecast_view(
    workflow,
    PREFERRED_MUNICIPALITY,
)
display_collapsible(
    f"Ver métricas de {municipality_view.municipality}",
    municipality_view.metrics,
)

forecast_figure = plot_municipality_forecast_evaluation(
    municipality_view.actual,
    municipality_view.forecasts,
    municipality_view.metrics,
    municipality=municipality_view.municipality,
)
display(
    HTML(
        forecast_figure.to_html(
            full_html=False,
            include_plotlyjs="cdn",
            config={"responsive": True, "displaylogo": False},
        )
    )
)


,Nombre Municipio,modelo,grupo_ingreso,mae_mm_clp,sesgo_mm_clp,suma_error_absoluto,suma_observado_absoluto,wape,meses_evaluados
0,Santiago,CatBoost global,FCM,706.864130,706.864130,4241.184780,8094.520781,0.523957,6.0
1,Santiago,CatBoost global,IPP,1833.419740,103.597036,11000.518440,61894.774353,0.177729,6.0
2,Santiago,CatBoost global,Otros ingresos,589.506105,-528.274272,3537.036632,8245.266491,0.428978,6.0
3,Santiago,CatBoost global,Total disponible,2264.699831,152.472703,13588.198989,87771.288501,0.154814,6.0
4,Santiago,CatBoost global,Transferencias corrientes,625.109653,221.386169,3750.657918,5490.978486,0.683058,6.0
5,Santiago,CatBoost global,Transferencias de capital,792.637649,-351.100360,4755.825894,4045.748390,1.175512,6.0
6,Santiago,ExtraTrees global,FCM,572.829991,283.483221,3436.979948,8094.520781,0.424606,6.0
7,Santiago,ExtraTrees global,IPP,1116.900185,-742.367150,6701.401111,61894.774353,0.108271,6.0
8,Santiago,ExtraTrees global,Otros ingresos,553.068757,-547.119136,3318.412544,8245.266491,0.402463,6.0
9,Santiago,ExtraTrees global,Total disponible,2161.219012,-1315.288180,12967.314072,87771.288501,0.147740,6.0
